[Reference](https://medium.com/@pankaj_pandey/2ad6c899d8ac)

In [1]:
!pip install blastai
!blastai serve

INFO: pip is looking at multiple versions of langchain to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of langchain-deepseek to determine which version is compatible with other requirements. This could t

object address  : 0x7c694d104ee0
object refcount : 3
object type     : 0xa2a4e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
Traceback (most recent call last):
^C


In [1]:
from openai import OpenAI
client = OpenAI(api_key="not-needed", base_url="http://127.0.0.1:8000")

In [4]:
stream = client.responses.create(
    model="not-needed",
    input="Compare fried chicken reviews for top 10 fast food restaurants",
    stream=True
)
for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta if " " in event.delta else "<screenshot>", end="", flush=True)

# Using the OpenAI-compatible surfaces


In [6]:
resp = client.chat.completions.create(
  model="not-needed",
  messages=[{"role":"user","content":"Find the 10 heaviest gorillas"}],
  stream=True
)
for chunk in resp:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="")

# Responses API

```
stream = client.responses.create(
  model="not-needed",
  input="Go to python.org and click Documentation",
  stream=True
)
for event in stream:
    # response.created, response.in_progress, response.output_text.delta, response.completed
    ...
```

# Performance playbook
```
constraints:
  allow_parallelism:
    task: true
    data: true
    first_of_n: false
  max_parallelism_nesting_depth: 1
```

```
constraints:
  llm_model: "openai:gpt-5"
  llm_model_mini: "openai:gpt-5-mini"
```

In [7]:
client.responses.create(
  model="not-needed",
  input="Search Python docs",
  cache_control="no-cache-plan"  # still uses results cache
)

# Example: Capped, parallel, cached “compare & rank” task

```
# config.yaml
settings:
  persist_cache: true
  logs_dir: "blast-logs/"
  blastai_log_level: "info"

constraints:
  max_concurrent_browsers: 4
  max_memory: "6GB"
  max_cost_per_minute: 0.25
  allow_parallelism:
    task: true
    data: true
```

In [8]:
from openai import OpenAI
client = OpenAI(api_key="not-needed", base_url="http://127.0.0.1:8000")

q = ("Compare laptop reviews for 5 models under ₹90k, "
     "score by display, battery and performance, then recommend top 2.")
stream = client.responses.create(model="not-needed", input=q, stream=True)
for ev in stream:
    if ev.type == "response.output_text.delta":
        if " " in ev.delta:  # thought/action
            print(ev.delta, end="", flush=True)
        else:                 # screenshot token
            print("<screenshot>", end="")